In [ ]:
import os

import torch
import torchaudio
from datasets import load_dataset
from pydub import AudioSegment
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor

os.chdir("/home/gpu/fuzzy/traffic_congestion")

print(os.getcwd())

In [ ]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(device)

In [ ]:
# Trim audio
voh_record = AudioSegment.from_wav("95.6_2022_11_30_100000_103000.wav")
clipped_vod_record = voh_record[: 5 * 1000]
clipped_vod_record.export("voh_record.wav", format="wav")
audio, sample_rate = torchaudio.load("voh_record.wav")
audio = torch.mean(audio, dim=0).unsqueeze(0)

In [ ]:
# Load model & processor
processor = Wav2Vec2Processor.from_pretrained("dragonSwing/wav2vec2-base-vietnamese")
model = Wav2Vec2ForCTC.from_pretrained("dragonSwing/wav2vec2-base-vietnamese")
model = model.to(device)
resampler = torchaudio.transforms.Resample(sample_rate, 16_000)

In [ ]:
inputs = processor(
    [resampler(audio).squeeze().numpy()],
    sampling_rate=16_000,
    return_tensors="pt",
    padding=True,
).to(device)

In [ ]:
with torch.no_grad():
    logits = model(inputs.input_values, attention_mask=inputs.attention_mask).logits
predicted_ids = torch.argmax(logits, dim=-1)
print("Prediction:", processor.batch_decode(predicted_ids))